In [1]:
from datetime import date

import pandas as pd
from sqlalchemy import func, or_, and_, not_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

### 1. Inizialize client and ORM session

In [2]:
patstat = PatstatClient(env="PROD")          # default: env="TEST"; use PatstatClient(env="PROD") for the full database
db = patstat.orm()                 # SQLAlchemy session used by all ORM queries below

### 2. Define multi-tier taxonomy and exlusion lists

In [3]:
# Primary Anchor Keywords (High Precision: Directly define flying UAV platforms)
DRONE_ANCHOR_KEYWORDS = [
    "%unmanned aerial vehicle%",
    "%unmanned aircraft%",
    "%unmanned aircraft system%",
    "%remotely piloted aircraft%",
    "%remotely piloted aircraft system%",
    "%uncrewed aerial vehicle%",
    "%autonomous aerial vehicle%",
    "%flying drone%",
    "%quadcopter%",
    "%multirotor%",
    "%multicopter%",
    "%hexacopter%",
    "%octocopter%",
    "%fixed-wing drone%",
    "%fixed-wing UAV%",
    "%hybrid VTOL%",
    "%micro air vehicle%",
    "% UAV %",      # Space-padded to avoid matching inside words
    "% RPAS %",     # Space-padded
]

# Secondary Functional Keywords (Subsystems, flight operations & equipment)
DRONE_SECONDARY_KEYWORDS = [
    "%drone%",
    "%flight control system%",
    "%autopilot system%",
    "%obstacle avoidance%",
    "%sense and avoid%",
    "%drone swarm%",
    "%multi-UAV coordination%",
    "%waypoint navigation%",
    "%unmanned traffic management%",
    "%tethered drone%",
    "%charging dock%",
]

# Exclusion Keywords (Filters out ground, marine, and toy domains)
DRONE_EXCLUSION_KEYWORDS = [
    "%toy%",
    "%underwater%",
    "%submersible%",
    "%marine%",
    "%underwater drone%",
    "%unmanned underwater vehicle%",
    "%unmanned ground vehicle%",
    "%acoustic drone%",
    "%surface vessel%",
    "%wind turbine%",
]


# 2. CPC CLASSIFICATION CODES (TLS224_APPLN_CPC)

# Core CPC Classes (B64U% automatically subsumes B64U10/00, B64U20/00, etc.)
DRONE_CORE_CPC_CODES = [
    "B64U%",        # Dedicated UAV subclass
    "B64C39/02%",   # Legacy UAV subclass
]

# Secondary CPC Classes (Subsystems, ground stations, and telemetry)
DRONE_SECONDARY_CPC_CODES = [
    "G05D1/00%",    # Flight / attitude control
    "G08G5/00%",    # Air traffic control / UTM
    "B64D47/00%",   # Aircraft equipment & gimbals
]


# 3. IPC CLASSIFICATION CODES (TLS209_APPLN_IPC)

# Core IPC Classes
DRONE_CORE_IPC_CODES = [
    "B64U%",
    "B64C39/02%",
]

# Secondary IPC Classes
DRONE_SECONDARY_IPC_CODES = [
    "G05D1/00%",
    "G08G5/00%",
]

### 3. Construct SQLAlchemy filter conditions

In [4]:
# 2. Build Atomic Filter Conditions
core_cpc_f = [TLS224_APPLN_CPC.cpc_class_symbol.like(c) for c in DRONE_CORE_CPC_CODES]
core_ipc_f = [TLS209_APPLN_IPC.ipc_class_symbol.like(c) for c in DRONE_CORE_IPC_CODES]

sec_cpc_f = [TLS224_APPLN_CPC.cpc_class_symbol.like(c) for c in DRONE_SECONDARY_CPC_CODES]
sec_ipc_f = [TLS209_APPLN_IPC.ipc_class_symbol.like(c) for c in DRONE_SECONDARY_IPC_CODES]

title_anchor_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_ANCHOR_KEYWORDS]
abstr_anchor_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_ANCHOR_KEYWORDS]

title_sec_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_SECONDARY_KEYWORDS]
abstr_sec_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_SECONDARY_KEYWORDS]

title_excl_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_EXCLUSION_KEYWORDS]
abstr_excl_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_EXCLUSION_KEYWORDS]
'''
applicant_f = [
    or_(TLS206_PERSON.doc_std_name.ilike(n), TLS206_PERSON.person_name.ilike(n))
    for n in LEONARDO_NAME_VARIANTS
]
'''

'\napplicant_f = [\n    or_(TLS206_PERSON.doc_std_name.ilike(n), TLS206_PERSON.person_name.ilike(n))\n    for n in LEONARDO_NAME_VARIANTS\n]\n'

In [7]:
# 3. Restructured Boolean Logic Clauses

# Boolean Clauses
standalone_core = or_(
    *core_cpc_f,
    *core_ipc_f,
    *title_anchor_f,
    *abstr_anchor_f
)

# Secondary codes MUST co-occur with explicit Anchor terms (not secondary keywords)
secondary_code_match = and_(
    or_(*sec_cpc_f, *sec_ipc_f),
    or_(*title_anchor_f, *abstr_anchor_f)
)

# Secondary keywords MUST co-occur with Core Codes
secondary_kw_match = and_(
    or_(*title_sec_f, *abstr_sec_f),
    or_(*core_cpc_f, *core_ipc_f)
)

# Master Conditions
inclusion_clause = or_(
    standalone_core,
    secondary_code_match,
    secondary_kw_match
)

exclusion_clause = not_(or_(*title_excl_f, *abstr_excl_f))

### 4. Build query, execute it and create DataFrame

In [8]:
MIN_FILING_DATE = date(2000, 1, 1)

query = (
    db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.docdb_family_id, # family ID
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_kind,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract,
        func.string_agg(func.distinct(TLS206_PERSON.doc_std_name), ', ').label("applicants"),
        func.string_agg(func.distinct(TLS224_APPLN_CPC.cpc_class_symbol), ', ').label("cpc_codes"),
        func.string_agg(func.distinct(TLS209_APPLN_IPC.ipc_class_symbol), ', ').label("ipc_codes"),
    )
    .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .outerjoin(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .outerjoin(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .filter(
        and_(
            inclusion_clause,
            exclusion_clause,
            # TLS207_PERS_APPLN.applt_seq > 0,
            # or_(*applicant_f),
            TLS201_APPLN.appln_filing_date >= MIN_FILING_DATE
        )
    )
    .group_by(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.docdb_family_id, # family ID
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_kind,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract
    )
)

In [9]:
results = query.all()
df = pd.DataFrame(results)

### data health check on the generated DataFrame 

In [10]:
# Missing text coverage
print("Abstract missingness:", df['appln_abstract'].isna().mean())
print("Title missingness:", df['appln_title'].isna().mean())

# Distinct count comparison
print("Total Applications (appln_id):", df['appln_id'].nunique())
print("Total Patent Families (docdb_family_id):", df['docdb_family_id'].nunique())

# Convert to datetime handling placeholder dates (e.g., 9999-12-31)
df['filing_year'] = pd.to_datetime(df['appln_filing_date'], errors='coerce').dt.year

# Check yearly distribution (excluding NaT)
print("Filing year distribution:")
print(df['filing_year'].value_counts().sort_index())

# Check count of invalid/placeholder dates
print("\nPlaceholder/Invalid date count:", df['filing_year'].isna().sum())

Abstract missingness: 0.0
Title missingness: 0.0
Total Applications (appln_id): 219997
Total Patent Families (docdb_family_id): 184577
Filing year distribution:
filing_year
2000       77
2001       87
2002      105
2003      130
2004      167
2005      200
2006      201
2007      245
2008      321
2009      439
2010      589
2011      653
2012      960
2013     1329
2014     2721
2015     6251
2016    13422
2017    19089
2018    20134
2019    20766
2020    23535
2021    22778
2022    21931
2023    24828
2024    25028
2025    14010
2026        1
Name: count, dtype: int64

Placeholder/Invalid date count: 0


In [11]:
print(f"Total matching patent applications retrieved: {len(df)}")
df.head()

Total matching patent applications retrieved: 219997


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_filing_date,appln_kind,appln_title,appln_abstract,applicants,cpc_codes,ipc_codes,filing_year
0,18996027,8847065,FR,0001923,2000-02-15,A,"ENGIN VOLANT TELECOMMANDE, EN PARTICULIER DE S...",The invention concerns a remote-controlled fly...,"BERTIN TECHNOLOGIES SA, CHASTANG GEORGES, LEPR...","B64U2101/30, B64U2201/20, B64U 50/11, B64U 5...","B64C 27/20, B64C 39/02",2000
1,21296989,10857873,WO,0002261,2000-06-09,W,HYDROGEN PEROXIDE BASED PROPULSION SYSTEM,A micro air vehicle comprising fuel tank conne...,"SECR DEFENCE, TILSTON JOHN RONALD, CHEUNG WAI SAN","F02K 9/68, F02K 9/74, F05D2250/82","F02K 9/74, B64C 39/00, F02K 9/68, F02K ...",2000
2,43103596,26655181,SE,0002650,2000-07-13,A,Unmanned aerial vehicle for low altitude opera...,The vehicle has a fixed vane rotor (1) directl...,"HJELTE FRITZ, AVIA ADVISER HB","B64C 29/02, B64C 27/06","B64C 29/02, B64C 27/06, B64C",2000
3,21298114,10857873,WO,0002837,2000-07-26,W,HYDROGEN PEROXIDE BASED PROPULSION SYSTEM,A micro air vehicle comprising fuel tank conne...,"TILSTON JOHN RONALD, SECR DEFENCE, CHEUNG WAI SAN","F02K 9/74, F02K 9/68, F05D2250/82","F02K 9/74, B64C 39/00, F02K 9/42, F02K ...",2000
4,18998227,9552048,WO,0003135,2000-11-10,W,HELICOPTER WITH HIGHLY STABLE AND HIGHLY MANOE...,The invention concerns a helicopter with pendu...,SALABERRY BERNARD DE,B64U 30/24,"B64C 27/10, B64C 27/52, B64C 27/48",2000


### save baseline DataFrame to disk to avoid re-querying PATSTAT for stage 2 (refinement via NLP)

In [13]:
# Save complete baseline sector dataset to parquet
df.to_parquet("flying_drones_baseline_prod.parquet", index=False)

# stage 2: semantic refinement 

### step 1: text preparation and corpus cleanup 

In [17]:
import pandas as pd

# 1. Load production dataset
df = pd.read_parquet("flying_drones_baseline_prod.parquet")

# 2. Clean text fields
df['appln_title'] = df['appln_title'].fillna('').str.strip()
df['appln_abstract'] = df['appln_abstract'].fillna('').str.strip()

# 3. Construct unified text column
df['full_text'] = df['appln_title'] + ". " + df['appln_abstract']

# 4. Identify and drop records with insufficient text
short_mask = df['full_text'].str.strip().str.len() <= 10
print(f"Removing {short_mask.sum()} records with missing/short text...")

df = df[~short_mask].copy().reset_index(drop=True)
print(f"Cleaned dataset total: {len(df)} valid records")

Removing 1 records with missing/short text...
Cleaned dataset total: 219996 valid records


# step 2: gold-standard seed patent selection 
Now that the corpus text is sanitized, run the seed extraction snippet to pull 10–20 high-confidence reference patents for Strategy A (Cosine Similarity against Centroid Vector)

In [19]:
%pip install pandas pyarrow numpy torch sentence-transformers scikit-learn tqdm
%pip install -q rank-bm25 sentence-transformers umap-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 41.7 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 32.1 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 76.6 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 68.4 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 43.8 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 83.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 49.6 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 16.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 29.0 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 2.9 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 40.3 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [25]:
!pip install -q gdown

import gdown

link = "https://drive.google.com/file/d/1HH02VISv9v5VGXevabLT6RZK5bRnXjZF/view?usp=drive_link"
file_id = "1HH02VISv9v5VGXevabLT6RZK5bRnXjZF" # Replace with your actual Google Drive file ID
output = "flying_drones_embeddings.npy"

gdown.download(id=file_id, output=output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1HH02VISv9v5VGXevabLT6RZK5bRnXjZF
From (redirected): https://drive.google.com/uc?id=1HH02VISv9v5VGXevabLT6RZK5bRnXjZF&confirm=t&uuid=f8580f6e-515e-4f98-8832-37eb8e54f4e2
To: /home/jovyan/innovation-management-datascience26/tutorial_0/flying_drones_embeddings.npy
100%|██████████| 338M/338M [00:01<00:00, 199MB/s]  


'flying_drones_embeddings.npy'

In [26]:
import os

file_path = "flying_drones_embeddings.npy"
file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
print(f"File size on disk: {file_size_mb:.2f} MB (Expected: ~322 - 338 MB)")

# Test load
corpus_embeddings = np.load(file_path)
print(f"Loaded successfully! Shape: {corpus_embeddings.shape}")

File size on disk: 339.26 MB (Expected: ~322 - 338 MB)
Loaded successfully! Shape: (219996, 384)


In [21]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
# 2. Load the uploaded embedding matrix
corpus_embeddings = np.load("flying_drones_embeddings.npy")

# Ensure 1:1 index alignment between DataFrame rows and embedding vectors
assert len(df) == corpus_embeddings.shape[0], f"Mismatch: {len(df)} DF rows vs {corpus_embeddings.shape[0]} embedding vectors!"
print(f"Successfully loaded embeddings matrix of shape {corpus_embeddings.shape}")

# 3. Load lightweight model on CPU (only used to encode the 15 seed texts in <1 second)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

# 4. Extract gold-standard seed candidates & compute seed embeddings
seed_candidates = df[
    df['cpc_codes'].str.contains('B64U', na=False) & 
    df['appln_title'].str.contains('unmanned aerial vehicle|quadcopter|multirotor', case=False, na=False)
].head(15)

seed_embeddings = model.encode(
    seed_candidates['full_text'].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True
)

# 5. Compute normalized seed centroid vector
centroid_vector = np.mean(seed_embeddings, axis=0, keepdims=True)
centroid_vector = centroid_vector / np.linalg.norm(centroid_vector)

# 6. Strategy A: Compute Cosine Similarity across all records (<1 sec on CPU)
df['sim_score_centroid'] = cosine_similarity(corpus_embeddings, centroid_vector).flatten()

# Print distribution quantiles
print("\nStrategy A Similarity Score Quantiles:")
print(df['sim_score_centroid'].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

Successfully loaded embeddings matrix of shape (219996, 384)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Strategy A Similarity Score Quantiles:
0.10    0.338747
0.25    0.438823
0.50    0.528157
0.75    0.600247
0.90    0.652173
0.95    0.679918
0.99    0.725442
Name: sim_score_centroid, dtype: float64


Key Insights from the Distribution
- Median (50th percentile = 0.528): Over half of the 220,000 baseline candidates score below ~0.53, confirming that a large portion of the initial SQL baseline consists of peripheral or indirect matches.
- 75th Percentile (0.600): A threshold of $\ge 0.60$ filters out 75% of the noise and retains the top 25% (~55,000 patents). This threshold provides a balanced candidate pool for high recall and solid precision.
- 90th Percentile (0.652): A strict threshold of $\ge 0.65$ isolates the top 10% (~22,000 patents), capturing high-precision, core drone system implementations.

Keep sim_score_centroid as a column in df without dropping any rows yet so it can be evaluated alongside and fused with subsequent strategies.

### lexical BM25 scores to compare traditional keyword ranking against centroid similarity

In [28]:
import re
from rank_bm25 import BM25Okapi

# 1. Whitespace & word-boundary tokenizer
def tokenize_text(text):
    return re.findall(r'\b\w+\b', text.lower())

print("Tokenizing full text corpus for BM25...")
tokenized_corpus = [tokenize_text(text) for text in df['full_text']]

# 2. Build BM25 index (~1-2 minutes on CPU)
bm25 = BM25Okapi(tokenized_corpus)

# 3. Construct rich multi-aspect domain query
bm25_query_str = (
    "unmanned aerial vehicle uav drone quadcopter multirotor flight control "
    "obstacle avoidance position course flight path autonomous traffic management utm"
)
tokenized_query = tokenize_text(bm25_query_str)

# 4. Compute raw BM25 scores
print("Calculating BM25 lexical scores across 220k records...")
df['bm25_score'] = bm25.get_scores(tokenized_query)

# 5. Min-Max normalize BM25 scores to [0, 1] for direct comparison/fusion with embeddings
bm25_min = df['bm25_score'].min()
bm25_max = df['bm25_score'].max()
df['bm25_norm'] = (df['bm25_score'] - bm25_min) / (bm25_max - bm25_min)

# 6. Compare top BM25 matches with Strategy A scores
print("\nTop 5 Lexical Matches:")
print(
    df.sort_values(by='bm25_score', ascending=False)[
        ['appln_id', 'appln_title', 'bm25_score', 'bm25_norm', 'sim_score_centroid']
    ].head()
)

Tokenizing full text corpus for BM25...
Calculating BM25 lexical scores across 220k records...

Top 5 Lexical Matches:
         appln_id                                        appln_title  \
212993  638070831  Unmanned aerial vehicle autonomous obstacle av...   
191231  628986485  Quad-rotor unmanned aerial vehicle autonomous ...   
191530  629801065  Unmanned aerial vehicle autonomous obstacle av...   
164049  604055034  Unmanned aerial vehicle autonomous flight path...   
190705  627333490  Unmanned aerial vehicle autonomous obstacle av...   

        bm25_score  bm25_norm  sim_score_centroid  
212993   54.009883   1.000000            0.422797  
191231   51.691243   0.957070            0.577611  
191530   51.435531   0.952336            0.619415  
164049   51.416468   0.951983            0.594547  
190705   51.067497   0.945521            0.557427  


The output reveals a classic divergence between lexical term density and dense semantic similarity.

Notice record 212993 at the top: it gets a perfect normalized BM25 score of 1.000 because its title contains almost every keyword from the query string ("Unmanned aerial vehicle autonomous obstacle avoidance..."), yet its sim_score_centroid is only 0.422—which falls below the 50th percentile of our embedding distribution. This happens because BM25 strongly rewards exact query word density, whereas Sentence Transformers capture the global conceptual balance of the full title and abstract text.

Conversely, record 191530 shows strong alignment across both methodologies, achieving bm25_norm = 0.952 and sim_score_centroid = 0.619 (top 20% of semantic similarity).

This contrast is precisely why combining lexical rankings with semantic embeddings yields a significantly better dataset than relying on either method alone.

In [29]:
df.head()

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_filing_date,appln_kind,appln_title,appln_abstract,applicants,cpc_codes,ipc_codes,filing_year,full_text,sim_score_centroid,bm25_score,bm25_norm
0,18996027,8847065,FR,0001923,2000-02-15,A,"ENGIN VOLANT TELECOMMANDE, EN PARTICULIER DE S...",The invention concerns a remote-controlled fly...,"BERTIN TECHNOLOGIES SA, CHASTANG GEORGES, LEPR...","B64U2101/30, B64U2201/20, B64U 50/11, B64U 5...","B64C 27/20, B64C 39/02",2000,"ENGIN VOLANT TELECOMMANDE, EN PARTICULIER DE S...",0.487131,0.000000,0.000000
1,21296989,10857873,WO,0002261,2000-06-09,W,HYDROGEN PEROXIDE BASED PROPULSION SYSTEM,A micro air vehicle comprising fuel tank conne...,"SECR DEFENCE, TILSTON JOHN RONALD, CHEUNG WAI SAN","F02K 9/68, F02K 9/74, F05D2250/82","F02K 9/74, B64C 39/00, F02K 9/68, F02K ...",2000,HYDROGEN PEROXIDE BASED PROPULSION SYSTEM. A m...,0.315246,3.492070,0.064656
2,43103596,26655181,SE,0002650,2000-07-13,A,Unmanned aerial vehicle for low altitude opera...,The vehicle has a fixed vane rotor (1) directl...,"HJELTE FRITZ, AVIA ADVISER HB","B64C 29/02, B64C 27/06","B64C 29/02, B64C 27/06, B64C",2000,Unmanned aerial vehicle for low altitude opera...,0.738386,13.932283,0.257958
3,21298114,10857873,WO,0002837,2000-07-26,W,HYDROGEN PEROXIDE BASED PROPULSION SYSTEM,A micro air vehicle comprising fuel tank conne...,"TILSTON JOHN RONALD, SECR DEFENCE, CHEUNG WAI SAN","F02K 9/74, F02K 9/68, F05D2250/82","F02K 9/74, B64C 39/00, F02K 9/42, F02K ...",2000,HYDROGEN PEROXIDE BASED PROPULSION SYSTEM. A m...,0.305451,3.492070,0.064656
4,18998227,9552048,WO,0003135,2000-11-10,W,HELICOPTER WITH HIGHLY STABLE AND HIGHLY MANOE...,The invention concerns a helicopter with pendu...,SALABERRY BERNARD DE,B64U 30/24,"B64C 27/10, B64C 27/52, B64C 27/48",2000,HELICOPTER WITH HIGHLY STABLE AND HIGHLY MANOE...,0.544910,3.600929,0.066672


### Strategy B: Unsupervised Clustering & Outlier Pruning (BERTopic / HDBSCAN)

Strategy B leverages BERTopic to group the 220,000 document vectors into thematic sub-clusters (e.g., Navigational Autopilot, Rotor Mechanics, UTM Air Traffic, Wind Turbines, Ground Robotics). This allows you to inspect cluster representations using class-based TF-IDF (c-TF-IDF) and drop non-relevant topics in bulk.

#### Step 1: Install Clustering Dependencies

In [30]:
%pip install bertopic umap-learn hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 149.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bertopic]1/2 [bertopic]
Note: you may need to restart the kernel to use updated packages.


#### Step 2: Fit BERTopic using Pre-Computed Embeddings
By explicitly passing your pre-computed corpus_embeddings matrix into fit_transform(), BERTopic bypasses the text encoding phase and executes clustering directly.

In [31]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# 1. Dimensionality reduction with UMAP (reduces 384 dimensions -> 5 for density clustering)
umap_model = UMAP(
    n_neighbors=15, 
    n_components=5, 
    min_dist=0.0, 
    metric='cosine', 
    random_state=42
)

# 2. Density-based clustering with HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=100,         # Minimum patents required to form a topic
    metric='euclidean', 
    cluster_selection_method='eom', 
    prediction_data=False
)

# 3. Topic keyword extraction via c-TF-IDF
vectorizer_model = CountVectorizer(stop_words="english", min_df=5, ngram_range=(1, 2))

# 4. Instantiate BERTopic
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    verbose=True
)

# 5. Run clustering using pre-computed embeddings matrix (~3-5 minutes on CPU)
print("Fitting BERTopic on pre-computed embeddings...")
topics, _ = topic_model.fit_transform(df['full_text'].tolist(), corpus_embeddings)

# Store assigned topic ID per patent (-1 represents HDBSCAN noise/outliers)
df['bertopic_cluster'] = topics

2026-11-01 17:37:09,161 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic on pre-computed embeddings...


2026-11-01 17:44:57,806 - BERTopic - Dimensionality - Completed ✓
2026-11-01 17:44:57,810 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-01 17:45:10,641 - BERTopic - Cluster - Completed ✓
2026-11-01 17:45:10,668 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-01 17:45:51,303 - BERTopic - Representation - Completed ✓


#### Step 3: Inspect & Filter Clusters
Once clustering completes, display the most frequent topics and their representative keywords:

In [32]:
# Display top 20 clusters by size
topic_info = topic_model.get_topic_info()
print("Top 20 Discovered Topics:")
print(topic_info.head(20)[['Topic', 'Count', 'Name']])

Top 20 Discovered Topics:
    Topic   Count                                               Name
0      -1  100131         -1_aerial_vehicle_unmanned_unmanned aerial
1       0    3614  0_inspection_line_power transmission_transmiss...
2       1    3051                 1_delivery_goods_express_logistics
3       2    2813  2_charging_wireless charging_charging device_v...
4       3    2587            3_crop_growth_vegetation_remote sensing
5       4    2474              4_drone_drones_drone control_location
6       5    2389  5_extinguishing_bomb_firefighting_extinguishin...
7       6    2013                 6_path_planning_path planning_task
8       7    1973                        7_folding_wing_arm_foldable
9       8    1755  8_surveying_surveying mapping_mapping_mapping ...
10      9    1750     9_pesticide_spraying_pesticide spraying_liquid
11     10    1746  10_construction_building_threedimensional_mapping
12     11    1564   11_control_control method_flight_vehicle control
13     1

To inspect specific topic details and identify topics for exclusion:

In [34]:
# Example: Inspect keywords for Topic 1
print("Keywords for Topic 1:")
print(topic_model.get_topic(-1))

# Example: Identify irrelevant clusters (e.g., wind turbines, ground chassis, acoustic sensors)
exclude_topic_ids = [3, 7, 12]  # Replace with actual non-drone topic IDs from your output

# Flag patents belonging to non-relevant clusters
df['is_excluded_cluster'] = df['bertopic_cluster'].isin(exclude_topic_ids)
print(f"Patents flagged for removal via cluster pruning: {df['is_excluded_cluster'].sum()}")

Keywords for Topic 1:
[('aerial', np.float64(0.004317666443997945)), ('vehicle', np.float64(0.004300339944572846)), ('unmanned', np.float64(0.004290284131609775)), ('unmanned aerial', np.float64(0.004215243500752519)), ('aerial vehicle', np.float64(0.004212540576640227)), ('control', np.float64(0.003839213642134013)), ('method', np.float64(0.003761156020648174)), ('flight', np.float64(0.003621190832435235)), ('device', np.float64(0.003532605212049634)), ('data', np.float64(0.00348203584698469))]
Patents flagged for removal via cluster pruning: 6093


### Workflow Progression Overview

#### Strategy C: Active Learning Classifier (Human-in-the-Loop)
Strategy C uses a supervised classifier (such as Logistic Regression or LightGBM) trained on a small set of labeled patent embeddings. The model learns the decision boundary between relevant UAV technology and irrelevant noise in the 384-dimensional vector space, predicting a probability score $P(\text{relevant})$ across all ~220,000 applications.

#### Step 1: Draw Sample & Annotation Setup

Run the following code to draw a 200-patent sample from your candidate pool. You can either manually label them in Pandas or use a high-confidence heuristic rule set to bootstrap the labels:

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# 1. Sample 200 patents randomly across the dataset
np.random.seed(42)
sample_indices = np.random.choice(len(df), size=200, replace=False)
sample_df = df.iloc[sample_indices].copy()

# 2. Bootstrap binary labels (1 = Relevant Flying UAV, 0 = Irrelevant/Noise)
# Note: For true manual active learning, export sample_df[['appln_id', 'appln_title', 'full_text']] 
# to CSV, annotate 1/0, and reload. Here we use high-confidence heuristic labeling to train:
is_core_cpc = sample_df['cpc_codes'].str.contains('B64U', na=False)
has_uav_title = sample_df['appln_title'].str.contains(
    'unmanned aerial|quadcopter|multirotor|drone|uav|flight control', case=False, na=False
)
sample_df['label'] = (is_core_cpc | has_uav_title).astype(int)

print(f"Sample label distribution (200 records):")
print(sample_df['label'].value_counts())

Sample label distribution (200 records):
label
1    166
0     34
Name: count, dtype: int64


##### Step 2: Train Classifier & Predict Full Corpus
Train a Logistic Regression model on the sample embedding vectors ($X \in \mathbb{R}^{200 \times 384}$) and predict relevance probabilities across the entire corpus_embeddings matrix ($X \in \mathbb{R}^{219996 \times 384}$):

In [36]:
# 1. Extract feature matrix for sample and full corpus
X_sample = corpus_embeddings[sample_indices]
y_sample = sample_df['label'].values

# 2. Instantiate and evaluate classifier using 5-Fold Cross-Validation
clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
cv_f1_scores = cross_val_score(clf, X_sample, y_sample, cv=5, scoring='f1')

print(f"5-Fold Cross-Validation F1-Score on Sample: {cv_f1_scores.mean():.4f} (+/- {cv_f1_scores.std():.4f})")

# 3. Fit classifier on full sample
clf.fit(X_sample, y_sample)

# 4. Predict probabilities across all ~220k vector embeddings (< 1 second on CPU)
df['prob_active_learning'] = clf.predict_proba(corpus_embeddings)[:, 1]

# 5. Display active learning probability distribution
print("\nStrategy C Probability Score Quantiles:")
print(df['prob_active_learning'].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

5-Fold Cross-Validation F1-Score on Sample: 0.9071 (+/- 0.0059)

Strategy C Probability Score Quantiles:
0.10    0.699453
0.25    0.780826
0.50    0.853373
0.75    0.898182
0.90    0.921657
0.95    0.931094
0.99    0.943777
Name: prob_active_learning, dtype: float64


### Multi-Strategy Summary Column Overview
- Metric Column: sim_score_centroid. Description: Strategy A: Cosine similarity to the seed centroid vector.
- Metric Column: bm25_norm. Description: Lexical BM25: Keyword density score (min-max normalized).
- Metric Column: bertopic_cluster. Description: Strategy B: Unsupervised topic ID (for bulk cluster exclusion).
- Metric Column: prob_active_learning. Description: Strategy C: Supervised classifier relevance probability.

## OPTION 1: Single-Strategy Filtering (Strategy A Strict Threshold) 

In [39]:
# Retain high-precision core set using Strategy A only
threshold = 0.65
df_cleaned_single = df[df['sim_score_centroid'] >= threshold].copy().reset_index(drop=True)

print(f"Single-Strategy Selection (sim >= {threshold}):")
print(f"Retained {len(df_cleaned_single):,} patents out of {len(df):,} baseline records.")

Single-Strategy Selection (sim >= 0.65):
Retained 23,112 patents out of 219,996 baseline records.


## OPTION 2: Hybrid Combination (Ensemble Filtering)

In [40]:
# Defined excluded non-drone clusters from Strategy B (e.g., wind turbines, ground chassis)
exclude_topic_ids = [-1]  # Update with specific non-drone cluster IDs from your output

# Combined multi-method threshold
df_cleaned_hybrid = df[
    (df['sim_score_centroid'] >= 0.60) &                 # Strategy A: Moderate semantic similarity
    (~df['bertopic_cluster'].isin(exclude_topic_ids)) &  # Strategy B: Strip non-relevant topic clusters
    (df['prob_active_learning'] >= 0.50)                 # Strategy C: Classifier probability pass
].copy().reset_index(drop=True)

print(f"\nHybrid Ensemble Selection:")
print(f"Retained {len(df_cleaned_hybrid):,} patents out of {len(df):,} baseline records.")


Hybrid Ensemble Selection:
Retained 29,248 patents out of 219,996 baseline records.


## once chosen a methodology, save the result

In [ ]:
# Save final clean dataset for Stage 3 technological analysis
df_cleaned_single.to_parquet("flying_drones_stage2_final.parquet", index=False)
print("Saved final cleaned corpus to flying_drones_stage2_final.parquet")

### applications in flying drones sector filed by Leonardo 

In [ ]:
# List of name variants covering current and historical Leonardo S.p.A. corporate entities
LEONARDO_NAME_VARIANTS = [
    "%LEONARDO%",
    "%FINMECCANICA%",
    "%AGUSTAWESTLAND%",
    "%AGUSTA%",
    "%SELEX%",
    "%ALENIA%",
    "%TELESPAZIO%",
]

# Construct applicant matching filters across standardized and raw person names
applicant_filters = [
    or_(
        TLS206_PERSON.doc_std_name.ilike(name),
        TLS206_PERSON.person_name.ilike(name)
    )
    for name in LEONARDO_NAME_VARIANTS
]

leonardo_query = (
    db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_kind,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract,
        func.string_agg(func.distinct(TLS206_PERSON.doc_std_name), ', ').label("applicants"),
        func.string_agg(func.distinct(TLS224_APPLN_CPC.cpc_class_symbol), ', ').label("cpc_codes"),
        func.string_agg(func.distinct(TLS209_APPLN_IPC.ipc_class_symbol), ', ').label("ipc_codes"),
    )
    .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .outerjoin(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .outerjoin(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .filter(
        and_(
            inclusion_clause,
            exclusion_clause,
            TLS207_PERS_APPLN.applt_seq_nr > 0,  # Restrict matches to applicants (excluding inventors)
            or_(*applicant_filters),
            # TLS201_APPLN.appln_filing_date >= MIN_FILING_DATE
        )
    )
    .group_by(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_kind,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract
    )
)

In [ ]:
leonardo_results = leonardo_query.all()
leonardo_df = pd.DataFrame(leonardo_results)

print(f"Total matching patent applications for Leonardo entities: {len(leonardo_df)}")
leonardo_df.head()